# SmartFood · Clasificación y ranking con SVM

Explorar un clasificador global de rating >= 4 con atributos de receta. Las predicciones y el baseline se interpretan automáticamente al evaluar; el ranking no aprende preferencias individuales. Consulta docs/evaluacion.md.

**Estado:** ejecución completa mediante el maestro el 6 de octubre de 2026. Los resultados son exploratorios; consulta el registro de ejecución y la evaluación para interpretar sus límites.

**Archivo:** `05_clasificacion_ranking_svm.ipynb`. Consulta la [guía de ejecución](../docs/notebooks.md) para conocer entradas, salidas y precauciones.


In [1]:
# Resolver la raíz del proyecto desde Jupyter o desde la carpeta del proyecto.
from pathlib import Path
import os
import sys

def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("smartfood")
if PROJECT_ROOT is None:
    raise RuntimeError("Inicia Jupyter desde la carpeta SmartFood o notebooks.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
for folder in ("data/intermediate", "data/processed", "data/output", "reports", "models"):
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

# Configuracion comun: registrar avisos para el resumen de alertas.
from smartfood.entorno_notebook import configurar_notebook
configurar_notebook()


## ✅ Paso 1: Carga de librerías

In [2]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline # , make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVC, LinearSVC
from smartfood.recomendaciones_svm import (
                                        recomendar_similares_por_receta, recomendar_por_keywords,
                                        recomendar_por_calorias, exportar_recomendaciones_svm_en_lote,
                                        ranking_recetas_nuevas, ranking_por_categoria,
                                        exportar_recomendaciones_svm_por_usuario,
                                        similitud_recetas_tfidf, evaluar_threshold
                                    )

## ✅ Paso 2: Carga de datos procesados

In [3]:
recipes = pd.read_parquet("data/processed/recipes_filtrado.parquet")
# TF-IDF requiere documentos de texto; Parquet conserva listas como arrays.
from smartfood.texto import texto_ingredientes
recipes["recipeingredientparts"] = recipes["recipeingredientparts"].map(texto_ingredientes)

reviews = pd.read_parquet("data/processed/reviews_filtrado.parquet")
print("Recipes:", recipes.shape)
print("Reviews:", reviews.shape)

Recipes: (190, 28)
Reviews: (10343, 8)


## ✅ Paso 3: Preparar dataset para SVM

Queremos predecir si un usuario **le gustó una receta**. Vamos a crear un target binario:
- 1 si `rating >= 4`
- 0 si `rating < 4`

In [4]:
reviews_bin = reviews.copy()
reviews_bin["target"] = (reviews_bin["rating"] >= 4).astype(int)

In [5]:
# Hacemos merge con los features de las recetas.

data = reviews_bin.merge(
    recipes[["recipeid", "calories", "aggregatedrating", "recipeingredientparts", "recipecategory"]],
    on="recipeid",
    how="left"
)


In [6]:
# Eliminamos registros sin ingredientes o calorías (opcional)

data.dropna(subset=["calories", "recipeingredientparts"], inplace=True)
print("Dataset final:", data.shape)

Dataset final: (10325, 13)


In [7]:
# Usamos solo las columnas necesarias
X_text = data["recipeingredientparts"]
X_num = data[["calories", "aggregatedrating"]]
y = data["target"]

In [8]:
X_cat = data["recipecategory"]

# Dividimos en entrenamiento y test (random_state fijo para reproducibilidad).
X_text_train, X_text_test, X_num_train, X_num_test, X_cat_train, X_cat_test, y_train, y_test = train_test_split(
    X_text, X_num, X_cat, y, test_size=0.2, random_state=42
)


## ✅ Paso 5: Entrenamiento del modelo SVM con Pipeline

In [9]:
# Seleccionamos las columnas según su tipo
columnas_numericas = ["calories", "aggregatedrating"]
columnas_categoricas = ["recipecategory"]
columna_texto = "recipeingredientparts"

# Definimos el ColumnTransformer con imputación incluida para numéricas
preprocesamiento = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),          # <-- aquí se imputan los nulos
        ("scaler", StandardScaler(with_mean=False))
    ]), columnas_numericas),
    
    ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
    
    ("text", TfidfVectorizer(), columna_texto)
], verbose_feature_names_out=False)

# Fusionamos texto y numéricos para entrenamiento
X_train_df = pd.concat([
    X_text_train.reset_index(drop=True).to_frame(name="recipeingredientparts"),
    X_num_train.reset_index(drop=True),
    X_cat_train.reset_index(drop=True).to_frame(name="recipecategory")
], axis=1)

X_test_df = pd.concat([
    X_text_test.reset_index(drop=True).to_frame(name="recipeingredientparts"),
    X_num_test.reset_index(drop=True),
    X_cat_test.reset_index(drop=True).to_frame(name="recipecategory")
], axis=1)

# Se usa LinearSVC (mas eficiente que SVC(kernel="linear") para este volumen de datos).
pipeline = Pipeline([
    ("pre", preprocesamiento),
    ("clf", LinearSVC())
])

pipeline.fit(X_train_df, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('pre', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['recipeingredientparts','calories','aggregatedrating','recipecategory']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_

## ✅ Paso 6: Evaluación del modelo

In [10]:
y_pred = pipeline.predict(X_test_df)
print("Reporte de clasificación:")
print(classification_report(y_test, y_pred, zero_division=0))
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       128
           1       0.94      1.00      0.97      1937

    accuracy                           0.94      2065
   macro avg       0.47      0.50      0.48      2065
weighted avg       0.88      0.94      0.91      2065


Matriz de confusión:
[[   0  128]
 [   0 1937]]


In [11]:
from IPython.display import Markdown, display
from smartfood.anotaciones import comentario_clasificacion
# Se calcula una sola vez y se reutiliza para mostrar y para registrar la etapa (evita duplicar el computo).
comentario = comentario_clasificacion(y_train, y_test, y_pred)
display(Markdown(comentario))

from smartfood.informe import registrar_etapa
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import numpy as np
clase_mayoritaria = int(np.unique(y_train, return_counts=True)[0][np.argmax(np.unique(y_train, return_counts=True)[1])])
metricas_clasificacion = {"casos": len(y_test), "baseline": {"accuracy": float(accuracy_score(y_test, np.full(len(y_test), clase_mayoritaria)))}, "modelo": {"accuracy": float(accuracy_score(y_test, y_pred)), "f1_macro": float(f1_score(y_test, y_pred, average="macro", zero_division=0)), "matriz_confusion": confusion_matrix(y_test, y_pred, labels=[0, 1]).tolist()}}
registrar_etapa(PROJECT_ROOT, "evaluacion", metricas_clasificacion, comentario, "05_clasificacion_ranking_svm.ipynb")


### Interpretación de clasificación

Accuracy: **93,80 %** sobre **2.065 casos**. El baseline constante de la clase mayoritaria de entrenamiento (**1**) obtiene **93,80 %** en el mismo test: el modelo iguala ese baseline en accuracy. Todas las predicciones pertenecen a la clase **1**.

Recall por clase: clase 0: **0,00 %** (128 casos); clase 1: **100,00 %** (1.937 casos). Si una clase no aparece en test, su recall no está definido. Revisa también F1 macro y la matriz de confusión; accuracy por sí sola no acredita discriminación. La comparación no mide calidad de ranking ni personalización: el clasificador utiliza atributos de receta.

## ✅ Paso 7: Generando recomendaciones

Recomendaciones por recetas

In [12]:
# buscamos un id de una receta.
recipe_id = recipes["recipeid"].iloc[100]  

# Ejecutar la función
recomendar_similares_por_receta(recipe_id, recipes, pipeline, top_n=10)



📌 Recetas similares a 'oreo balls':
32. bananachocolate chip muffins (Score: 1.00)
129. pumpkin spice bread (Score: 0.92)
94. basic pancake mix (Score: 0.92)
184. soft peanut butter cookies (Score: 0.91)
71. mashed red potatoes with garlic and parmesan (Score: 0.88)
4. taco seasoning mix (Score: 0.88)
31. chicken parmesan (Score: 0.87)
147. southwestern baked spaghetti (Score: 0.87)
159. award winning chili (Score: 0.87)
109. turkey meatloaf (Score: 0.87)


,recipeid,name,authorid,authorname,cooktime,preptime,totaltime,datepublished,description,images,...,cholesterolcontent,sodiumcontent,carbohydratecontent,fibercontent,sugarcontent,proteincontent,recipeservings,recipeyield,recipeinstructions,score
31,165,bananachocolate chip muffins,27395,manda,pt20m,pt15m,pt35m,2002-02-11 09:39:00+00:00,wow get thisa very lowfat muffin recipe that a...,[https://img.sndimg.com/food/image/upload/w_55...,...,15.8,315.2,37.5,1.3,22.2,2.8,12.0,12 mufins,"[Preheat oven to 350 degrees., Lightly coat 12...",1.000000
128,123,pumpkin spice bread,103628,jmelyn,pt1h10m,pt20m,pt1h30m,2003-09-26 19:03:00+00:00,hooray for fall we love this pumpkin bread rec...,[https://img.sndimg.com/food/image/upload/w_55...,...,31.0,279.0,40.2,1.2,25.7,3.1,24.0,2 loaves,"[Heat oven to 350., in a large mixing bowl, co...",0.921234
93,148,basic pancake mix,27678,sackville,pt15m,pt15m,pt30m,2002-12-13 20:01:00+00:00,say goodbye to aunt jemima once you taste thes...,[https://img.sndimg.com/food/image/upload/w_55...,...,138.2,937.9,61.8,1.7,8.7,14.0,NaN,8 large pancakes,"[In a bowl, mix together all the dry ingredien...",0.915862
183,93,soft peanut butter cookies,166642,cookingontheside,pt7m,pt10m,pt17m,2007-01-16 22:51:00+00:00,this is my motherinlaws recipeshe was always b...,[https://img.sndimg.com/food/image/upload/w_55...,...,28.7,154.6,23.6,0.8,14.6,3.7,30.0,4 dozen,"[Cream butter, peanut butter and both sugars t...",0.906845
70,99,mashed red potatoes with garlic and parmesan,27643,mizznezz,pt25m,pt5m,pt30m,2002-07-16 22:41:00+00:00,make and share this mashed red potatoes with g...,[https://img.sndimg.com/food/image/upload/w_55...,...,16.7,529.4,31.7,3.2,2.5,6.0,6.0,None,"[Put potatoes and garlic in lg pan., Cover wit...",0.875490
3,180,taco seasoning mix,1539,opal fitzgerald,None,pt5m,pt5m,1999-08-15 07:45:00+00:00,make and share this taco seasoning mix recipe ...,[https://img.sndimg.com/food/image/upload/w_55...,...,0.0,2372.5,6.0,1.5,1.5,0.9,NaN,1 package,[Combine all ingredients in a small bowl and b...,0.875380
30,69,chicken parmesan,27643,mizznezz,pt15m,pt8m,pt23m,2002-02-07 17:10:00+00:00,make and share this chicken parmesan recipe fr...,[https://img.sndimg.com/food/image/upload/w_55...,...,176.8,963.5,19.8,2.4,6.9,36.4,4.0,None,"[Pound chicken to flatten., Salt and pepper to...",0.871919
146,154,southwestern baked spaghetti,37036,kim d,pt30m,pt30m,pt1h,2004-03-20 19:59:00+00:00,this is an easy yet very tasty spaghetti casse...,[https://img.sndimg.com/food/image/upload/w_55...,...,103.5,785.1,37.0,3.2,5.7,26.9,6.0,None,"[Cook spaghetti per package direcions., Combin...",0.871091
158,91,award winning chili,141293,kelbel,pt2h,pt15m,pt2h15m,2004-12-11 19:59:00+00:00,my husband and i created this recipe together ...,[https://img.sndimg.com/food/image/upload/w_55...,...,60.5,1355.7,34.2,11.0,6.8,27.0,6.0,None,[In a medium Dutch oven over medium-high heat ...,0.869704
108,120,turkey meatloaf,66050,tootyflooty,pt55m,pt10m,pt1h5m,2003-02-24 20:03:00+00:00,this recipe came from a package of ground turk...,[https://img.sndimg.com/food/image/upload/w_55...,...,127.7,973.4,21.1,1.2,10.5,26.2,5.0,None,"[Melt butter in a skillet., Cook onion and gar...",0.869536


Recomendaciones por ingredientes

In [13]:
ingredientes = "chicken, garlic, lemon, olive oil"
# ingredientes = "spicy chicken garlic"

# Ejecutar la función
recomendar_por_keywords(ingredientes, pipeline, recipes, top_n=5)


📌 Recomendaciones basadas en texto: 'chicken, garlic, lemon, olive oil'
109. turkey meatloaf (Score: 1.00)
128. country white bread or dinner rolls bread machine (Score: 0.99)
118. buttery bread machine rolls (Score: 0.99)
127. pizza dough for thin crust pizza (Score: 0.98)
117. romanos macaroni grill rosemary bread (Score: 0.97)


,recipeid,name,authorid,authorname,cooktime,preptime,totaltime,datepublished,description,images,...,cholesterolcontent,sodiumcontent,carbohydratecontent,fibercontent,sugarcontent,proteincontent,recipeservings,recipeyield,recipeinstructions,score
108,120,turkey meatloaf,66050,tootyflooty,pt55m,pt10m,pt1h5m,2003-02-24 20:03:00+00:00,this recipe came from a package of ground turk...,[https://img.sndimg.com/food/image/upload/w_55...,...,127.7,973.4,21.1,1.2,10.5,26.2,5.0,None,"[Melt butter in a skillet., Cook onion and gar...",1.000000
127,92,country white bread or dinner rolls bread machine,60260,ddw7976,pt15m,pt30m,pt45m,2003-09-14 20:01:00+00:00,we think this is the best white bread recipe e...,[https://img.sndimg.com/food/image/upload/w_55...,...,12.4,238.7,24.3,0.9,3.4,3.5,15.0,None,"[In bread machine pan, place all ingredients f...",0.991878
117,141,buttery bread machine rolls,11044,jean1490,pt2h,pt10m,pt2h10m,2003-06-23 20:01:00+00:00,these golden fluffy butterfy rolls are great h...,[https://img.sndimg.com/food/image/upload/w_55...,...,27.1,190.6,18.6,0.7,2.1,3.2,NaN,24 rolls,"[In bread machine pan, put all ingredients in ...",0.991832
126,87,pizza dough for thin crust pizza,154044,unknownchef86,pt15m,pt1h,pt1h15m,2003-09-02 20:02:00+00:00,notethis is one of the recipes i adopted in th...,[https://img.sndimg.com/food/image/upload/w_55...,...,0.0,299.2,120.7,5.2,0.4,17.6,NaN,2 pizzas,"[Mix a little sugar into the warm water., Spri...",0.976136
116,178,romanos macaroni grill rosemary bread,88877,hunter,pt18m,pt2h15m,pt2h33m,2003-06-12 20:04:00+00:00,make and share this romanos macaroni grill ros...,[https://img.sndimg.com/food/image/upload/w_55...,...,30.5,1274.4,128.4,6.1,6.7,18.7,NaN,2 loaves,"[Place yeast, sugar and water in large bowl or...",0.972640


Recomendaciones por calorias

In [14]:
recomendar_por_calorias(350, recipes, top_n=5)


📌 Recetas más cercanas a 350 calorías:
180. fantastic taco casserole (349.9 cal)
179. paneras cream cheese potato soup (349.6 cal)
165. baked chicken (351.1 cal)
30. zucchini bread (347.6 cal)
55. ovenfried chicken chimichangas (353.9 cal)


,recipeid,name,authorid,authorname,cooktime,preptime,totaltime,datepublished,description,images,...,cholesterolcontent,sodiumcontent,carbohydratecontent,fibercontent,sugarcontent,proteincontent,recipeservings,recipeyield,recipeinstructions,diferencia
0,177,fantastic taco casserole,29667,leilani,pt10m,pt20m,pt30m,2006-02-07 20:16:00+00:00,i originally found this taco casserole recipe ...,[https://img.sndimg.com/food/image/upload/w_55...,...,72.8,1190.0,16.2,5.3,3.1,24.1,NaN,None,"[Brown ground beef and drain., Add taco season...",0.1
1,27,paneras cream cheese potato soup,279367,shawn h,pt45m,pt15m,pt1h,2006-01-08 20:02:00+00:00,this is my favorite recipe for potato soup its...,[https://img.sndimg.com/food/image/upload/w_55...,...,62.5,938.0,29.6,3.4,3.7,11.4,NaN,None,"[Combine broth, potatoes, and spices., Boil on...",0.4
2,156,baked chicken,37036,kim d,pt50m,pt5m,pt55m,2005-02-21 19:59:00+00:00,this chicken recipe is so easy i hesitated in ...,[https://img.sndimg.com/food/image/upload/w_55...,...,123.3,192.7,0.0,0.0,0.0,30.4,4.0,None,"[Preheat oven to 400°F., Spray a shallow bakin...",1.1
3,106,zucchini bread,27678,sackville,pt1h,pt10m,pt1h10m,2002-01-12 17:39:00+00:00,this moist bread has been in the family for as...,[https://img.sndimg.com/food/image/upload/w_55...,...,34.9,136.3,50.4,1.1,31.8,3.8,16.0,2 loaves,"[Preheat oven to 350 degrees., Combine all ing...",2.4
4,19,ovenfried chicken chimichangas,6258,miss annie,pt25m,pt20m,pt45m,2002-05-13 18:11:00+00:00,a fast and healthier way to serve chimichangas...,[https://img.sndimg.com/food/image/upload/w_55...,...,46.0,717.9,31.5,2.4,2.2,18.7,NaN,6 chimichangas,"[Mix chicken, picante sauce or salsa, cumin, o...",3.9


Exportar recomendaciones para todos los usuarios

In [15]:
exportar_recomendaciones_svm_en_lote(
    pipeline=pipeline,
    df_reviews=reviews,
    df_recipes=recipes,
    top_n=5,
    max_users=30
)

🔄 Generando recomendaciones:   0%|          | 0/30 [00:00<?, ?it/s]

🔄 Generando recomendaciones:  67%|██████▋   | 20/30 [00:00<00:00, 192.38it/s]

🔄 Generando recomendaciones: 100%|██████████| 30/30 [00:00<00:00, 195.54it/s]


✅ Recomendaciones exportadas a data/output/recomendaciones_svm.csv (150 filas)


👤 Usuarios con recomendaciones: 30
🍽️ Recetas recomendadas únicas: 7
📊 Promedio de recetas por usuario: 5.00


Ranking de recetas nuevas

In [16]:
ranking_recetas_nuevas(pipeline, recipes, top_n=5)


📌 Ranking de 5 recetas nuevas recomendadas por el modelo:
32. bananachocolate chip muffins (Score: 1.00)
129. pumpkin spice bread (Score: 0.92)
94. basic pancake mix (Score: 0.92)
184. soft peanut butter cookies (Score: 0.91)
71. mashed red potatoes with garlic and parmesan (Score: 0.88)


,recipeid,name,authorid,authorname,cooktime,preptime,totaltime,datepublished,description,images,...,cholesterolcontent,sodiumcontent,carbohydratecontent,fibercontent,sugarcontent,proteincontent,recipeservings,recipeyield,recipeinstructions,score
0,165,bananachocolate chip muffins,27395,manda,pt20m,pt15m,pt35m,2002-02-11 09:39:00+00:00,wow get thisa very lowfat muffin recipe that a...,[https://img.sndimg.com/food/image/upload/w_55...,...,15.8,315.2,37.5,1.3,22.2,2.8,12.0,12 mufins,"[Preheat oven to 350 degrees., Lightly coat 12...",1.000000
1,123,pumpkin spice bread,103628,jmelyn,pt1h10m,pt20m,pt1h30m,2003-09-26 19:03:00+00:00,hooray for fall we love this pumpkin bread rec...,[https://img.sndimg.com/food/image/upload/w_55...,...,31.0,279.0,40.2,1.2,25.7,3.1,24.0,2 loaves,"[Heat oven to 350., in a large mixing bowl, co...",0.921234
2,148,basic pancake mix,27678,sackville,pt15m,pt15m,pt30m,2002-12-13 20:01:00+00:00,say goodbye to aunt jemima once you taste thes...,[https://img.sndimg.com/food/image/upload/w_55...,...,138.2,937.9,61.8,1.7,8.7,14.0,NaN,8 large pancakes,"[In a bowl, mix together all the dry ingredien...",0.915862
3,93,soft peanut butter cookies,166642,cookingontheside,pt7m,pt10m,pt17m,2007-01-16 22:51:00+00:00,this is my motherinlaws recipeshe was always b...,[https://img.sndimg.com/food/image/upload/w_55...,...,28.7,154.6,23.6,0.8,14.6,3.7,30.0,4 dozen,"[Cream butter, peanut butter and both sugars t...",0.906845
4,99,mashed red potatoes with garlic and parmesan,27643,mizznezz,pt25m,pt5m,pt30m,2002-07-16 22:41:00+00:00,make and share this mashed red potatoes with g...,[https://img.sndimg.com/food/image/upload/w_55...,...,16.7,529.4,31.7,3.2,2.5,6.0,6.0,None,"[Put potatoes and garlic in lg pan., Cover wit...",0.875490


Recomendaciones por categoría o keywords

In [17]:
ranking_por_categoria("dessert", pipeline, recipes, top_n=5)


📌 Ranking de recetas en la categoría 'dessert':
184. soft peanut butter cookies (Score: 1.00)
50. chocolate chocolate chocolate bundt cake with chocolate glaze (Score: 0.95)
122. big grandmas best peanut butter cookies (Score: 0.95)
138. the most wonderful gingerbread cookies (Score: 0.90)
116. soft molasses cookies (Score: 0.84)


,recipeid,name,authorid,authorname,cooktime,preptime,totaltime,datepublished,description,images,...,cholesterolcontent,sodiumcontent,carbohydratecontent,fibercontent,sugarcontent,proteincontent,recipeservings,recipeyield,recipeinstructions,score
0,93,soft peanut butter cookies,166642,cookingontheside,pt7m,pt10m,pt17m,2007-01-16 22:51:00+00:00,this is my motherinlaws recipeshe was always b...,[https://img.sndimg.com/food/image/upload/w_55...,...,28.7,154.6,23.6,0.8,14.6,3.7,30.0,4 dozen,"[Cream butter, peanut butter and both sugars t...",1.000000
1,102,chocolate chocolate chocolate bundt cake with ...,24235,ccinsc,pt50m,pt15m,pt1h5m,2002-05-03 10:49:00+00:00,need i say morechocolate decadently rich and m...,[https://img.sndimg.com/food/image/upload/w_55...,...,27.1,393.3,51.4,2.5,36.0,3.9,16.0,1 cake,"[Cake: Preheat oven to 350°., Grease and flour...",0.950886
2,161,big grandmas best peanut butter cookies,86627,liisan,pt12m,pt15m,pt27m,2003-07-15 20:00:00+00:00,this is my husbands grandmothers recipe that i...,[https://img.sndimg.com/food/image/upload/w_55...,...,13.5,141.0,18.3,0.6,12.2,2.9,36.0,3 dozen cookies,"[Cream together the sugars, Crisco, peanut but...",0.945692
3,53,the most wonderful gingerbread cookies,72483,gingerkitten d,pt8m,pt2h15m,pt2h23m,2004-01-05 20:00:00+00:00,this is my very favorite gingerbread cookie re...,[https://img.sndimg.com/food/image/upload/w_55...,...,15.4,94.6,24.4,0.6,10.7,1.9,NaN,24 5inch tall cookies,"[In a small bowl, whisk together flour, baking...",0.901583
4,138,soft molasses cookies,82616,shirl j 831,pt10m,pt30m,pt40m,2003-06-06 20:02:00+00:00,ok your cookie jar will be so empty when you m...,[https://img.sndimg.com/food/image/upload/w_55...,...,4.4,92.5,16.4,0.3,9.7,0.9,42.0,42 cookies,"[Cream together shortening and brown sugar., S...",0.836894


Similitud entre recetas usando TF-IDF

In [18]:
# resultados = similitud_recetas_tfidf(recipes)
# replace=False evita recipeid duplicados en la muestra (romperian la busqueda por clave en "similitudes").
recipes_sample = recipes.sample(n=min(5000, len(recipes)), random_state=42, replace=False).reset_index(drop=True)
similitudes = similitud_recetas_tfidf(recipes_sample)

In [19]:
recipe_id = recipes_sample['recipeid'].sample(n=1).values[0]
similitudes[recipe_id]

,recipeid,name,score
53,27,paneras cream cheese potato soup,0.511651
84,66,buffalo chicken dip,0.418176
71,32,crock pot cream cheese chicken,0.390550
139,132,uncle bills microwave potato chips,0.386243
146,110,loaded baked potato salad,0.377931


Evaluar distintos thresholds de decisión

Probar con decision_function valores > 0, > 0.3, etc., para ajustar sensibilidad del modelo.

In [20]:
evaluar_threshold(pipeline, X_test_df, y_test)


🔎 Threshold: 0
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       128
           1       0.94      1.00      0.97      1937

    accuracy                           0.94      2065
   macro avg       0.47      0.50      0.48      2065
weighted avg       0.88      0.94      0.91      2065


🔎 Threshold: 0.2
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       128
           1       0.94      1.00      0.97      1937

    accuracy                           0.94      2065
   macro avg       0.47      0.50      0.48      2065
weighted avg       0.88      0.94      0.91      2065


🔎 Threshold: 0.4
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       128
           1       0.94      1.00      0.97      1937

    accuracy                           0.94      2065
   macro avg       0.47      0.50      0.48      2065
weighted avg       0.8

Recomendaciones por usuario

In [21]:
exportar_recomendaciones_svm_por_usuario(pipeline, reviews, recipes, path="data/output/recomendaciones_svm.csv", top_n=5, max_users = 100)


✅ Recomendaciones exportadas a data/output/recomendaciones_svm.csv


,usuario,receta_id,receta_nombre,score
0,11,165,bananachocolate chip muffins,1.00
1,11,123,pumpkin spice bread,0.92
2,11,148,basic pancake mix,0.92
3,11,93,soft peanut butter cookies,0.91
4,11,99,mashed red potatoes with garlic and parmesan,0.88
...,...,...,...,...
495,322,165,bananachocolate chip muffins,1.00
496,322,123,pumpkin spice bread,0.92
497,322,148,basic pancake mix,0.92
498,322,93,soft peanut butter cookies,0.91


## Alertas de la ejecución

La tabla agrupa los avisos de Python y conserva su detalle original. Si una celda falla antes de llegar aquí, puedes ejecutar esta celda de resumen para consultar las alertas ya registradas. Los errores y los mensajes de diagnóstico impresos no se incluyen automáticamente.


In [22]:
from smartfood.entorno_notebook import mostrar_alertas
mostrar_alertas(PROJECT_ROOT / "reports" / "alertas" / "05_clasificacion_ranking_svm.json")
from smartfood.informe import generar_informe
generar_informe(PROJECT_ROOT)


WindowsPath('D:/portafolio/Analisis/Recommenders/SmartFood/reports/informe_final/informe_final.md')